# AudioMNIST ±1 LSB matching %5/%10 — CNN v4: topluluk + TTA (T4 GPU)

**Çalışma zamanı → T4 GPU**, sonra **Tümünü çalıştır**.

- v3 verisi ve rolleri aynı (34 eğitim / 16 kalibrasyon / 10 test konuşmacısı); v3'ün Drive checkpoint'lerini kullanır.
- %10 için iki yeni üye v3'ün %20 checkpoint'inden eğitilir (yeni seed; ikincisi 32k kırpma). Her biri ~1 saat.
- Aday skorlayıcılar (tek model / topluluk, TTA'lı ya da TTA'sız) yalnızca kalibrasyonda, bir konuşmacı dışarıda bırakılarak karşılaştırılır; %80 kapısı.
- Test bir kez çalışır. v3 test konuşmacılarına **ikinci bakış** olduğu için sonuç adaptif kanıttır.
- Çıktılar `MyDrive/steg-audiomnist-cnn-v4`; kopma olursa **Tümünü çalıştır** kaldığı yerden devam eder.


In [ ]:
import os

import torch
from google.colab import drive

drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/steg-audiomnist-cnn-v3"
OUT = "/content/drive/MyDrive/steg-audiomnist-cnn-v4"
if not os.path.exists(os.path.join(BASE, "best-0.2.pt")):
    raise SystemExit("v3 checkpoint yok: " + BASE)
os.makedirs(OUT, exist_ok=True)
if not torch.cuda.is_available():
    raise SystemExit("GPU yok: Çalışma zamanı türünü T4 GPU yap ve yeniden çalıştır")
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
%%bash
set -euo pipefail
SHA=708e377aaa4c498a3aab7cad84940bbbd34dabc3
if [ ! -d /content/steganography/.git ]; then
  git init -q /content/steganography
  git -C /content/steganography remote add origin https://github.com/erkanrzgc/steganography
fi
cd /content/steganography
git fetch -q --depth 1 origin "$SHA"
git checkout -q FETCH_HEAD
test "$(git rev-parse HEAD)" = "$SHA"
git log -1 --format='%h %s'

In [ ]:
%%bash
set -euo pipefail
cd /content
URL=$(python -c "import json; print(json.load(open('/content/steganography/notebooks/audiomnist_wav_cnn_layout_v3_20261121.json'))['url'])")
if [ ! -f audiomnist.zip ]; then
  curl -sSL --fail --retry 3 -o audiomnist.part "$URL"
  mv audiomnist.part audiomnist.zip
fi
ls -l audiomnist.zip

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
OUT=/content/drive/MyDrive/steg-audiomnist-cnn-v3
cd /content/steganography
python -m research.wav_cnn_audiomnist prepare --layout notebooks/audiomnist_wav_cnn_layout_v3_20261121.json \
  --data /content/amnist-v3 --archive /content/audiomnist.zip

## Üyelerin eğitimi

Her satır bir epoch: üye, kayıp, kalibrasyon BA ve AUC.


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
BASE=/content/drive/MyDrive/steg-audiomnist-cnn-v3
OUT=/content/drive/MyDrive/steg-audiomnist-cnn-v4
cd /content/steganography
python -m research.wav_cnn_audiomnist_ensemble train --layout notebooks/audiomnist_wav_cnn_layout_v3_20261121.json \
  --data /content/amnist-v3 --base "$BASE" --out "$OUT" --device cuda


## Seçim (yalnızca kalibrasyon)


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
BASE=/content/drive/MyDrive/steg-audiomnist-cnn-v3
OUT=/content/drive/MyDrive/steg-audiomnist-cnn-v4
cd /content/steganography
python -m research.wav_cnn_audiomnist_ensemble freeze --layout notebooks/audiomnist_wav_cnn_layout_v3_20261121.json \
  --data /content/amnist-v3 --base "$BASE" --out "$OUT" --device cuda


## Tek seferlik test (adaptif ikinci bakış)

Kalibrasyonda %80'in altında kalan oranlar test edilmez. Tekrar çalıştırmak hata verir, bu beklenen davranıştır.


In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
BASE=/content/drive/MyDrive/steg-audiomnist-cnn-v3
OUT=/content/drive/MyDrive/steg-audiomnist-cnn-v4
cd /content/steganography
if [ ! -f "$OUT/choice-freeze.json" ]; then
  echo "Seçim dondurulmadı; test atlandı."
elif [ -f "$OUT/test-result.json" ]; then
  echo "Test zaten bir kez çalıştı; sonuç korunuyor."
else
  python -m research.wav_cnn_audiomnist_ensemble test --layout notebooks/audiomnist_wav_cnn_layout_v3_20261121.json \
    --data /content/amnist-v3 --base "$BASE" --out "$OUT" --device cuda
fi


In [ ]:
import json
import os
import zipfile

from google.colab import files

archive = "/content/audiomnist-cnn-v4-results.zip"
with zipfile.ZipFile(archive, "w") as bundle:
    for name in ["choice-freeze.json", "test-result.json", "seed-b/history.json",
                 "seed-b/best.json", "crop-32k/history.json", "crop-32k/best.json"]:
        path = os.path.join(OUT, name)
        if os.path.exists(path):
            bundle.write(path, name)
for name in ["seed-b/best.json", "crop-32k/best.json", "test-result.json"]:
    path = os.path.join(OUT, name)
    if os.path.exists(path):
        with open(path) as stream:
            print(name, json.dumps(json.load(stream), indent=1)[:1500])
path = os.path.join(OUT, "choice-freeze.json")
if os.path.exists(path):
    with open(path) as stream:
        frozen = json.load(stream)
    for rate, choice in frozen["choices"].items():
        print(rate, choice["selected"], choice["passes_gate"],
              [(r["candidate"], round(r["loso_ba"], 4), round(r["loso_worst"], 4))
               for r in choice["candidates"]])
files.download(archive)
